# Hugging face -> Executorch -> 4 bit quantization -> Lower to Metal backend -> export to .pte

In [1]:
# (1) show python and os info
import sys, platform
print("python:", sys.version)
print("platform:", platform.platform())


python: 3.11.2 (v3.11.2:878ead1ac1, Feb  7 2023, 10:02:41) [Clang 13.0.0 (clang-1300.0.29.30)]
platform: macOS-15.6.1-arm64-arm-64bit


In [2]:
import importlib, pkgutil
def show(pkg):
    try:
        m = importlib.import_module(pkg)
        import inspect
        v = getattr(m, '__version__', None)
        print(f"{pkg}: version {v}")
    except Exception as e:
        print(f"{pkg}: NOT INSTALLED ({e})")
for p in ("torch","executorch","torchao","optimum","optimum_executorch","transformers"):
    show(p)

torch: version 2.11.0.dev20251222
executorch: version None


W0210 20:41:13.699000 93847 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


torchao: version 0.16.0+git026b76d12
optimum: version None
optimum_executorch: NOT INSTALLED (No module named 'optimum_executorch')


/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: version 5.0.0rc1


In [3]:
"""
git clone https://github.com/huggingface/optimum-executorch.git
cd optimum-executorch
python install_dev.py 

"""

'\ngit clone https://github.com/huggingface/optimum-executorch.git\ncd optimum-executorch\npython install_dev.py \n\n'

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import nn

MODEL_ID = "Qwen/Qwen3-1.7B"
SEQ_LEN = 1024
DTYPE = torch.float16

torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="cpu",
    trust_remote_code=True,
)

# Force static cache internally
base_model.config.use_cache = False
base_model.config.cache_implementation = "static"
base_model.config.attn_implementation = "eager"  #Other types use a mask which becomes None on export
base_model.model.rotary_emb.cache_max_seq_len = 0
base_model.model.rotary_emb._set_cos_sin_cache = False



base_model.eval()

cfg = base_model.config

NUM_LAYERS = cfg.num_hidden_layers
NUM_HEADS = cfg.num_attention_heads
HEAD_DIM = cfg.hidden_size // cfg.num_attention_heads
SEQ_LEN = 1024

past_key = torch.zeros(
    (NUM_LAYERS, 1, NUM_HEADS, SEQ_LEN, HEAD_DIM),
    dtype=torch.float16,
)

past_value = torch.zeros(
    (NUM_LAYERS, 1, NUM_HEADS, SEQ_LEN, HEAD_DIM),
    dtype=torch.float16,
)

input_ids = torch.zeros((1, 1), dtype=torch.long)

#print(NUM_LAYERS, NUM_HEADS, HEAD_DIM)
#print(past_key, past_value, input_ids)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:04<00:00, 76.94it/s, Materializing param=model.norm.weight]                               
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
# 🔑 WRAPPER: removes cache from outputs

class Qwen3KVWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.config = model.config

    def forward(
        self,
        input_ids: torch.Tensor,
        past_key: torch.Tensor,
        past_value: torch.Tensor,
    ):
        # Build a fake static cache from tensors
        # This replaces DynamicCache entirely
        from transformers.cache_utils import StaticCache

        batch_size = input_ids.shape[0]

        cache = StaticCache(
            config=self.config,
            max_batch_size=batch_size,
            max_cache_len=past_key.shape[2],
            device=input_ids.device,
            dtype=past_key.dtype,
        )

        # Inject tensor KV into cache
        for i in range(len(cache.key_cache)):
            cache.key_cache[i].copy_(past_key[i])
            cache.value_cache[i].copy_(past_value[i])


        outputs = self.model(
            input_ids=input_ids,
            past_key_values=cache,
            use_cache=True,
            return_dict=False,
        )

        logits, new_cache = outputs

        # Extract updated KV tensors
        new_key = new_cache.key_cache
        new_value = new_cache.value_cache

        return logits, new_key, new_value
# ^^^ Ignore above function   

class Qwen3ForExecuTorch(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        
        attention_mask = torch.ones_like(input_ids)

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask, 
            use_cache=False,
            return_dict=False,   # IMPORTANT
        )
        # outputs = (logits, past_key_values)
        logits = outputs[0]
        return logits


model = Qwen3ForExecuTorch(base_model)
model.eval()

#print(model)


Qwen3ForExecuTorch(
  (model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layer

In [6]:
import torch
from torch.export import export

# Create sample input for tracing (match Qwen3's expected input shape)
example_inputs = (torch.zeros((1, SEQ_LEN), dtype=torch.long),) 

# Export to a GraphModule
exported_model = export(model, example_inputs)
#print("GRAPH SIGNATURE:")
#print(exported_model.graph_signature)

W0210 20:41:23.118000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.119000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.120000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.120000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.121000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.122000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.123000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0210 20:41:23.123000 93847 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels


In [7]:
## Check exported model

x = torch.randint(0, 151_936, (1, SEQ_LEN), dtype=torch.long)
print(exported_model.module()(x).shape) # torch.Size([1, 1024, 151936])

## Forward signature

import inspect
print(inspect.signature(model.forward)) # (input_ids)


## Dry run

with torch.no_grad():
    #out = model(input_ids, past_key, past_value)
    out = model(input_ids)

print(type(out), len(out)) # (tuple, 3)

## Shape check

#logits, new_key, new_value = out

logits = out

print(type(logits))
      # type(new_key), type(new_value)) # torch.Tensor
print(logits.shape) # (1, 1, vocab_size)
#print(new_key.shape, past_key.shape) # equal
#print(new_value.shape, past_value.shape) #equal


torch.Size([1, 1024, 151936])
(input_ids)
<class 'torch.Tensor'> 1
<class 'torch.Tensor'>
torch.Size([1, 1, 151936])


## Quantization 1

In [ ]:
#Again does not work

import torch
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer 
from torchao.quantization.pt2e.quantizer import QuantizationConfig, QuantizationSpec
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e
from torch.ao.quantization.observer import PlaceholderObserver
from torch.ao.quantization import FakeQuantize

# Initialize quantizer for 4-bit weight-only
# This will result in (int8 activations + int4 weights) or weight-only patterns
quantizer = XNNPACKQuantizer()


activation_spec = QuantizationSpec(
    dtype=torch.int8,                    # activation mapping to quantized dtype
    qscheme=torch.per_tensor_affine,
    is_dynamic=False,
    observer_or_fake_quant_ctr=MinMaxObserver
)

# 2. Manually define the 4-bit weight-only spec

weight_spec = QuantizationSpec(
    dtype=torch.int8,
    #bit_width=4,
    qscheme=torch.per_channel_symmetric,
    is_dynamic=False, # Weights are static
    observer_or_fake_quant_ctr= PerChannelMinMaxObserver
)


quant_config = QuantizationConfig(
    input_activation=activation_spec,
    output_activation=activation_spec,
    weight=weight_spec,
    bias=None
)

#print(quant_config)

quantizer = XNNPACKQuantizer()
quantizer.set_global(quant_config)

#print(quantizer)
# 3. Prepare (Insert Observers/Fake Quantize)
# This step inserts "fake" quantization nodes that simulate 4-bit behavior

prepared_model = prepare_pt2e(exported_model.module(), quantizer) #This doesn't support int4??


In [ ]:
with torch.no_grad():
    out = prepared_model(input_ids)
print(out.shape)

In [ ]:
print("Starting calibration...")

with torch.no_grad():
    for _ in range(8):
       ids = torch.randint(0, 151_936, (1, SEQ_LEN), dtype=torch.long)
       prepared_model(ids)
print("Calibration complete.")

In [ ]:
quantized_model = convert_pt2e(prepared_model)
print(quantized_model)

## Quantization 2

In [8]:
### This approach introduces dequantize_affine from torchao which cannot be handled by Metal backend

import torch
from torchao.quantization import quantize_
#from torchao.quantization.quant_api import IntxWeightOnlyConfig
from torchao.quantization.quant_api import Int8WeightOnlyConfig, Int4CPULayout


int8_config = Int8WeightOnlyConfig(
    #group_size=32, 
    #layout=Int4CPULayout(),
    version=2, # to avoid fbgemm-gen-ai
)

quantize_(
    model,
    int8_config,
)

In [9]:
quantized_exported_program = torch.export.export(
    model,
    #args=(input_ids, past_key, past_value),
    args=(input_ids,),
    strict=False, 
)


In [11]:
# Verify 
print("Type of program:")
print(type(quantized_exported_program)) #torch.export.ExportedProgram
print("------")

## Graph signature
print("Graph signature:")

sig = quantized_exported_program.graph_signature

print("User inputs:", sig.user_inputs) # input_ids past_key past_value
print("Outputs:", sig.output_specs) # logits new_key new_value
print("------")

## No python objects

print("No python objects:")
for spec in sig.output_specs: # torch.export.TensorSpec
    print(type(spec.arg))
print("------")

## Check for int4 weights

print("int8 quantization:")
gm = quantized_exported_program.graph_module   # len > 0, quantized_linear, int4_linear, dequantize_per_channel

graph = quantized_exported_program.graph

print(f"{'Node Name':<45} | {'Inner Tensor Info'}")
print("-" * 100)

for node in graph.nodes:
    # Filter for the specific operator used by torchao subclasses
    if "access_subclass_inner_tensor" in str(node.target):
        
        # Extract the metadata (the bit you see in quotes in readable())
        # This is stored in node.meta['val'] for the fake tensor properties
        meta_val = node.meta.get('val', "N/A")
        
        # Target arguments: the first arg is the weight, second is the 'string' identifier
        # like 'tensor_impl' or 'packed_weight'
        attr_type = node.args[1] if len(node.args) > 1 else "unknown"

        print(f"{node.name:<45} | {attr_type:<15} | {meta_val}")

#quant_ops = [
#    n.target for n in gm.graph.nodes
#    if "quant" in str(n.target).lower()
#       or "dequant" in str(n.target).lower()
#]

#print(len(quant_ops), quant_ops[:10])

print("------")

## Check forward execution

print("Forward execution:")

with torch.no_grad():
    #logits, new_key, new_value = quantized_exported_program.module()(
    logits = quantized_exported_program.module()(
        input_ids,
        #input_ids, past_key, past_value
    )

print(logits.shape) #(1, 1, vocab_size)
#print(new_key.shape, past_key.shape) # equal
#print(new_value.shape, past_value.shape) # equal
print("------")

## KV cache is enabled
print("Cache config:")
print("use_cache =", base_model.config.use_cache) #True
print("cache_implementation =", base_model.config.cache_implementation) #static
print("------")

## Static graph
print("Static graph:")
print(quantized_exported_program.range_constraints) #empty dict/ full resolved ranges? No symbolic dimensions
print("------")

## No training ops
print("No training ops:")
print(quantized_exported_program.graph_module.training) # False
print("------")

## Memory check
print("Memory check:")
import io, torch

buf = io.BytesIO()
torch.save(quantized_exported_program.state_dict, buf)
print(buf.tell() / (1024 ** 3), "GB - might be inflated")
print("------")

## Does it have complex torchao ops?
print("torchao.dequantize_affine:")  #Yes?
gm = quantized_exported_program.graph_module

torchao_ops = [
    n for n in gm.graph.nodes
    if "torchao" in str(n.target)
]

print(len(torchao_ops), torchao_ops[:10])
print("------")

# This prints the ATen IR graph code
#quantized_exported_program.graph_module.print_readable()

Type of program:
<class 'torch.export.exported_program.ExportedProgram'>
------
Graph signature:
User inputs: ('input_ids',)
Outputs: [OutputSpec(kind=<OutputKind.USER_OUTPUT: 1>, arg=TensorArgument(name='to_826'), target=None)]
------
No python objects:
<class 'torch.export.graph_signature.TensorArgument'>
------
int8 quantization:
Node Name                                     | Inner Tensor Info
----------------------------------------------------------------------------------------------------
access_subclass_inner_tensor_default_2        | qdata           | FakeTensor(..., size=(2048, 2048), dtype=torch.int8)
access_subclass_inner_tensor_default_3        | scale           | FakeTensor(..., size=(2048, 1), dtype=torch.float16)
access_subclass_inner_tensor_default_6        | qdata           | FakeTensor(..., size=(1024, 2048), dtype=torch.int8)
access_subclass_inner_tensor_default_7        | scale           | FakeTensor(..., size=(1024, 1), dtype=torch.float16)
access_subclass_inner_

In [ ]:
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.apple.mps.partition import MPSPartitioner

# 1. Define the config to allow the specialized CPU op to pass to the partitioner
edge_config = EdgeCompileConfig(
    _core_aten_ops_exception_list=[
        torch.ops.aten._weight_int4pack_mm_for_cpu.default
    ]
)

# 2. Lower everything in one go
# This internally calls to_edge and then delegates to Metal
edge_program = to_edge_transform_and_lower(
    quantized_exported_program,
    partitioner=[MPSPartitioner(compile_specs = [])],
    compile_config=edge_config
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
[DEBUG 2026-02-10 20:42:45,739 mps_partitioner.py:48] [UNSUPPORTED] Node aten.alias_copy.default not supported
[DEBUG 2026-02-10 20:42:45,740 mps_partitioner.py:48] [UNSUPPORTED] Node aten.any.dim not supported
[DEBUG 2026-02-10 20:42:45,741 mps_partitioner.py:48] [UNSUPPORTED] Node scalar_tensor.default not supported
[DEBUG 2026-02-10 20:42:45,749 mps_partitioner.py:48] [UNSUPPORTED] Node scalar_tensor.default not supported
[DEBUG 2026-02-10 20:42:45,750 mps_partitioner.py:48] [UNSUPPORTED] Node dim_

In [ ]:
edge_program.dump_partitioning()

In [ ]:
# Convert to executorch program

executorch_program = lowered_program.to_executorch()

# Approach 2 - Direct executorch API

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Qwen/Qwen3-1.7B', local_dir='models/qwen3_1_7b', local_dir_use_symlinks=False)

In [ ]:
# Step 1: Convert weights to the official ExecuTorch format
"""
python -m executorch.examples.models.qwen3.convert_weights \
   "models/qwen3_1_7b" \
    models/qwen3_1_7_b_converted.bin

curl -L -o models/1_7b_config.json https://raw.githubusercontent.com/pytorch/executorch/main/examples/models/qwen3/config/1_7b_config.json

git clone org-21003710@github.com:pytorch/executorch.git

git submodule sync
git submodule update --init --recursive

./install_executorch.sh --editable

inside executorch

export PYTHONPATH=$PYTHONPATH:$(pwd):$(pwd)/examples/models/llama

export FLATC_EXECUTABLE=$(which flatc)

export EX_PATH=$(pwd)
export PYTHONPATH=$EX_PATH:$PYTHONPATH

# Get the absolute path to your executorch repo
export EX_ROOT=$(pwd)

# Force the specific source sub-directories into your path
export PYTHONPATH=$EX_ROOT:$EX_ROOT/examples/models/llama:$PYTHONPATH

export FLATC_EXECUTABLE=$(find $VIRTUAL_ENV -name flatc -type f -perm +111 | head -n 1)

"""


"""
# Add to export_llama
import torchao
import executorch.kernels.quantized # This triggers the AOT registration for out-variants
from executorch.exir.passes import DecomposeQuantizedOpsPass # Add this at top

...

# ... inside the main export logic ...
edge_manager = exir.to_edge(exported_program)

# --- ADD THIS LINE --- _export_llama()
# This decomposes torchao ops into basic math that the memory planner understands
    if builder.edge_manager is not None:
        builder.edge_manager = builder.edge_manager.transform(DecomposeQuantizedOpsPass())

-----

//python3 -c "import torchao; torchao.ops.load_all()"

pip install --pre torchao --extra-index-url https://download.pytorch.org/whl/nightly/cpu

# Run this from the executorch root
//./install_executorch.sh --pybind mps

//python3 -m executorch.examples.models.llama.export_llama --help | grep -E "quant|decom"

export EXECUTORCH_BUILD_TORCHAO=ON
export EXECUTORCH_BUILD_MPS=ON

PYTHON_EXECUTABLE=$(which python3) ./install_executorch.sh

"""
# from inside executorch 
# Step 2: Use the official export script for Metal/MPS
"""
python -m executorch.examples.models.llama.export_llama \
    --model "qwen3_1_7b" \
    --checkpoint ../qwen3_export/models/qwen3_1_7_b_converted.bin \
    --params ../qwen3_export/models/1_7b_config.json \
    --output_name ../qwen3_export/models/qwen3_1_7B_1024_int4_metal.pte \
    --mps \
    --quantization_mode "4w" \
    --max_seq_length 1024 \
    --max_context_length 1024 \
    --disable_dynamic_shape \
    --use_kv_cache
    


    --use-torchao-kernels



    --use_kv_cache

"""

#choices = ["int8", "8da4w", "8da4w-gptq", "4w"] - 8da4w is a more complex scheme. MPS fails

# Misc code

In [29]:
# Check

import torch
import torchao
try:
    # Check if the specific op exists in the torchao namespace
    print("Checking for torchao ops...")
    print(f"Dequantize Affine exists: {hasattr(torch.ops.torchao, 'dequantize_affine')}")
except Exception as e:
    print(f"Ops not found: {e}")

Checking for torchao ops...
Dequantize Affine exists: True


In [31]:
#Check

import torch
import executorch.kernels.quantized # Important!
print(torch.ops.torchao.dequantize_affine.out is not None)

AttributeError: The underlying op of 'torchao.dequantize_affine' has no overload name 'out'

In [ ]:
from executorch.exir import EdgeCompileConfig
from executorch.exir import to_edge
from executorch.backends.apple.mps.partition import MPSPartitioner
from executorch.exir.backend.backend_api import to_backend

# Convert to Edge IR
edge_program = to_edge(
    quantized_exported_program,
    compile_config=EdgeCompileConfig(
        _check_ir_validity=True,
        _core_aten_ops_exception_list=[
            torch.ops.aten._weight_int4pack_mm_for_cpu.default]
    ),
    #decompositions=torchao_decompositions,

)

print("✅ Converted to Edge IR")

# Metal partitioning
from executorch.backends.apple.mps.partition import MPSPartitioner

mps_partitioner = MPSPartitioner(
    compile_specs=[]
)

lowered_program = edge_program.to_backend(
    partitioner=mps_partitioner,
)

print("✅ MPS partitioning applied")


/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/torchao/dtypes/uintx/int4_cpu_layout.py:82: UserWarning: Models quantized with version 1 of Int4WeightOnlyConfig is deprecated and will no longer be supported in a future release, please upgrade torchao and quantize again, or download a newer torchao checkpoint, see https://github.com/pytorch/ao/issues/2948 for more details
  warnings.warn(
/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/torchao/dtypes/uintx/int4_cpu_layout.py:82: UserWarning: Models quantized with version 1 of Int4WeightOnlyConfig is deprecated and will no longer be supported in a future release, please upgrade torchao and quantize again, or download a newer torchao checkpoint, see https://github.com/pytorch/ao/issues/2948 for more details
  warnings.warn(
/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/torchao/dtypes/uintx/int4_cpu_layout.py:82: UserWarning: Models quantized with version 1 of Int4WeightOnlyConfig is deprecated a

✅ Converted to Edge IR


In [ ]:
# CoreML partitioning

from executorch.exir import to_edge_transform_and_lower
from executorch.backends.apple.coreml.partition import CoreMLPartitioner
from executorch.backends.apple.coreml.compiler import CoreMLBackend
import coremltools as ct


coreml_partitioner = CoreMLPartitioner(
    compile_specs = CoreMLBackend.generate_compile_specs(
        minimum_deployment_target=ct.target.iOS16
    )
)

lowered_program = edge_program.to_backend(
    partitioner=coreml_partitioner,
)
print("✅ CoreML partitioning succeeded")

In [ ]:

from torchao.utils import unwrap_tensor_subclass

# Unwrap torchao tensor subclasses to decompose high-level ops into core ATen ops
# This ensures the graph uses only standard math (mul, add, bitwise) that Metal supports.

unwrapped_module = unwrap_tensor_subclass(quantized_exported_program.module())

quantized_exported_program = torch.export.export(
    unwrapped_module,
    #args=(input_ids, past_key, past_value),
    args=(input_ids,),
    strict=False, 
)

# This table tells ExecuTorch how to turn "dequantize_affine" into "mul/sub/add"
# We create a decomposition registry for the specific torchao op
from torchao.quantization.quant_primitives import dequantize_affine

def dequantize_affine_wrapper(
    input_tensor,  # index 0
    block_size_input,
    scale,
    zero_point, 
    in_dtype,       
    quant_min,     
    quant_max,  
    out_dtype,   
):
    # Fix the FunctionalTensor issue
    #actual_block_size = block_size_input.tolist() #if isinstance(block_size, torch.Tensor) else None
    #in_dtype = input_tensor.dtype
    # Explicitly call using names to ensure the functional logic is clear
    return dequantize_affine(
        input=input_tensor,
        block_size=block_size_input,
        scale=scale,
        zero_point=zero_point,
        input_dtype=in_dtype,
        quant_min=quant_min,
        quant_max=quant_max,
        output_dtype=out_dtype,
    )

ao_decompositions = {
    torch.ops.torchao.dequantize_affine.default: dequantize_affine_wrapper,
}

quantized_exported_program =quantized_exported_program.run_decompositions(ao_decompositions)

print("✅ Quantized PT2E export succeeded")

# Use the underscore prefix for OpOverload objects
schema = torch.ops.torchao.dequantize_affine.default._schema
print(f"Full Schema: {schema}")

# Let's break down the arguments specifically
print("\nArguments expected by the Graph:")
for i, arg in enumerate(schema.arguments):
    print(f"Index [{i}]: Name='{arg.name}' | Type={arg.type}")

In [22]:
import inspect
from executorch.backends.apple.mps.partition import MPSPartitioner

print(inspect.signature(edge_program.to_backend))

print(inspect.getsource(edge_program.to_backend))

(*args: Any, **kwargs: Any) -> Any
            def wrapper(*args: Any, **kwargs: Any) -> Any:
                return func(*args, **kwargs)



In [ ]:
"""
git clone https://github.com/pytorch/executorch.git
cd executorch
git checkout v1.1.0
git submodule update --init --recursive


If required:

pip uninstall -y executorch
rm -rf build dist *.egg-info
xcrun --sdk macosx --show-sdk-path

export EXECUTORCH_BUILD_METAL=1
export CMAKE_ARGS="-DEXECUTORCH_BUILD_METAL=ON"
export USE_PYBIND=1

export PIP_USE_PEP517=0
export SETUPTOOLS_USE_DISTUTILS=stdlib

export PYTHON_EXECUTABLE="$(which python)"
export Python_EXECUTABLE="$(which python)"

rm -rf build
rm -f CMakeCache.txt
rm -rf CMakeFiles

TORCH_BASE=$(python - <<'EOF'
import torch, os
print(os.path.dirname(torch.__file__))
EOF
)
echo $TORCH_BASE




EXPORT_METAL=1 python setup.py develop # with metal backend
"""